# Практика. Дообучение T5 с помощью LoRA на датасете SQuAD

Это завершающий урок по практике для моделей encoder-decoder. Вы уже реализовали решения задач перевода и NER. В этом уроке вы реализуете QA — модель, которая будет отвечать на вопрос по контексту. Как и NER, эту задачу можно решить и энкодером, то есть для каждого токена определить, является он частью ответа на вопрос или нет. Это простое, но негибкое решение: явного ответа в контексте может не оказаться, ответ может быть косвенным или его может быть нужно получить из нескольких фактов. С такими ситуациями энкодер не справляется. Генеративные модели лишены этих недостатков — если их достаточно обучить, они выдают ответ в любом формате, могут интерпретировать знания из контекста и доставить их из своей «памяти».


## 1. Датасет
Мы поработаем с русскоязычным аналогом датасета SQUAD — [SberQUAD](https://huggingface.co/datasets/kuznetsoffandrey/sberquad). Чтобы решить задачу seq2seq-подходом, нам пригодятся всего три поля: `context`, `question`, `answers`. 

Предобработаем этот датасет, как для обычной seq2seq-задачи. В этом уроке мы будем использовать русскоязычную T5 — [ruT5-base](https://huggingface.co/ai-forever/ruT5-base). 

### Задание 1
Напишите функцию для предобработки и токенизации входных данных и ответа. Это могут быть разные функции или одна. Для экономии времени оставьте по 10% от train и validation выборок. 

Для экономии вычислений обрежьте последовательность до зафиксированных значений: вопрос+контекст — 512 токенов, ответ — 64. 

Контекст и вопрос склейте в одну строку любым способом.

In [1]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer


MODEL_ID = "ai-forever/ruT5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

MAX_SRC_LEN = 512
MAX_TGT_LEN = 64

raw = load_dataset("kuznetsoffandrey/sberquad")


def tokenize_split(dataset, split):
    records = []
    data = dataset[split].train_test_split(test_size=0.1, seed=42)['test']
    for ex in data:
        context = ex["context"]
        question = ex["question"]
        task = tokenizer(f"{context}\n{question}", padding="max_length", truncation=True, max_length=MAX_SRC_LEN)["input_ids"]
        answer = tokenizer(ex["answers"]["text"][0], padding="max_length", truncation=True, max_length=MAX_TGT_LEN)["input_ids"]
        records.append({"task": task, "answer": answer})
    return Dataset.from_list(records)

tokenized_train = tokenize_split(raw, "train")
tokenized_val = tokenize_split(raw, "validation")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Как надо

In [2]:
def format_example(ex):
    q = ex["question"].strip()
    c = ex["context"].strip()
    y = ex["answers"]["text"][0].strip() if ex["answers"]["text"] else ""
    src = f"context: {c} \nquestion: {q}"
    tgt = y
    return {"input_text": src, "labels_text": tgt}


def tokenize(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_SRC_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        batch["labels_text"],
        max_length=MAX_TGT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels_ids = labels["input_ids"]
    labels_ids = [[(tid if tid != tokenizer.pad_token_id else -100) for tid in seq] for seq in labels_ids]
    model_inputs["labels"] = labels_ids
    return model_inputs


formatted_train = raw["train"].shard(num_shards=10, index=0).map(format_example)
formatted_val = raw["validation"].shard(num_shards=10, index=0).map(format_example)

cols_to_remove = [c for c in raw["train"].column_names if c not in ["input_text", "labels_text"]]
formatted_train = formatted_train.remove_columns(cols_to_remove)
formatted_val = formatted_val.remove_columns(cols_to_remove)

tokenized_train = formatted_train.map(tokenize, batched=True, remove_columns=["input_text", "labels_text"])
tokenized_val = formatted_val.map(tokenize, batched=True, remove_columns=["input_text", "labels_text"])

print(tokenized_train[0].keys()) 

dict_keys(['input_ids', 'attention_mask', 'labels'])


## 2. Метрики
Модели будем сравнивать классическими метриками для этой задачи: EM (Exact match) — доля строк с полным совпадением), F1 — баланс точности и полноты. Поскольку это метрики для задачи SQuAD, их можно посчитать метрикой `squad` из библиотеки `evaluate` — `evaluate.load(“squad“)`.  

Дополнительно используем метрику CER (Char Error Rate) — доля неправильных символов. Метрика CER удобна для генеративных моделей: она не штрафует слово целиком, если оно частично неверно написано, например, неправильно только окончание. 

Реализуем CER через расстояние Левенштейна — минимальное количество символов, которые нужно изменить в первой строке, чтобы получить вторую. Минимальная реализация выглядит так:

In [3]:
import Levenshtein

def compute_cer(preds, refs):
    errors = 0
    lens = 0
    for p, r in zip(preds, refs):
        errors += Levenshtein.distance(p, r)
        lens += len(r)
    return round(errors / lens * 100, 2) # количество ошибок на длину правильных текстов

print(compute_cer(['Привт'], ['Привет']))

16.67


## Задание 2
Реализуйте функцию, которая посчитает и вернёт все метрики, описанные выше. Используйте `evaluate`.


In [4]:
import torch
import evaluate
import numpy as np

squad_metric = evaluate.load("squad_v2")

def compute_metrics_with_evaluate(preds, refs):
    ids = list(range(len(refs)))
    predictions = [{
        "id": str(id),
        "prediction_text": pred,
        "no_answer_probability": 0.,
        } for id, pred in zip(ids, preds)]
    references = [{
        "id": str(id),
        "answers": {
            "text": [ref],
            "answer_start": [0],
        },
        } for id, ref in zip(ids, refs)]
    results = squad_metric.compute(predictions=predictions, references=references)
    cer = compute_cer(preds, refs)
    return {
        "EM": results["exact"],
        "F1": results["f1"],
        "CER": cer,
        "count": len(preds)
    }

In [5]:
compute_metrics_with_evaluate(["Привет"], ["Привет!"])

{'EM': 100.0, 'F1': 100.0, 'CER': 14.29, 'count': 1}

## 3. Файнтюн полной модели
Дообучим модель с помощью `Seq2SeqTrainer`. Код файнтюна на эту задачу аналогичен коду для других seq2seq-задач.

### Задание 3
Допишите гиперпараметры для обучения и обучите модель. После обучения реализуйте декодирование предсказанных и правильных токенов в текст, чтобы посчитать метрику.

In [6]:
import torch
from transformers import AutoModelForSeq2SeqLM, \
                        DataCollatorForSeq2Seq, \
                        Seq2SeqTrainingArguments, \
                        Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="rut5_base_full",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=1,
    eval_strategy="epoch",
    predict_with_generate=True,
    gradient_accumulation_steps=1,
    fp16=True,
    optim='adafactor', # специально придуманный оптимизатор для T5
)


trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

torch.cuda.empty_cache()

trainer.train()

res = trainer.predict(tokenized_val)

preds = res.predictions
preds[preds < 0] = 0
pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
labels = np.array(tokenized_val['labels'])
labels[labels < 0] = 0
label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

print(compute_metrics_with_evaluate(pred_texts, label_texts))

Epoch,Training Loss,Validation Loss
1,0.261200,0.478693


{'EM': 61.111111111111114, 'F1': 80.36537914861847, 'CER': 33.56, 'count': 504}


In [18]:
idx = 0
print(tokenizer.decode(tokenized_val[idx]["input_ids"], skip_special_tokens=True))
print(pred_texts[idx])
print(label_texts[idx])

context: Первые упоминания о строении человеческого тела встречаются в Древнем Египте. В XXVII веке до н. э. египетский врач Имхотеп описал некоторые органы и их функции, в частности головной мозг, деятельность сердца, распространение крови по сосудам. В древнекитайской книге Нейцзин (XI—VII вв. до н. э.) упоминаются сердце, печень, лёгкие и другие органы тела человека. В индийской книге Аюрведа ( Знание жизни , IX-III вв. до н. э.) содержится большой объём анатомических данных о мышцах, нервах, типах телосложения и темперамента, головном и спинном мозге. question: Где встречаются первые упоминания о строении человеческого тела?
в Древнем Египте
в Древнем Египте


In [23]:
idx = 12
print(tokenizer.decode(tokenized_val[idx]["input_ids"], skip_special_tokens=True))
print(pred_texts[idx])
print(label_texts[idx])

context: Сверхкороткие импульсы лазерного излучения используются в лазерной химии для запуска и анализа химических реакций. Здесь лазерное излучение позволяет обеспечить точную локализацию, дозированность, абсолютную стерильность и высокую скорость ввода энергии в систему. В настоящее время разрабатываются различные системы лазерного охлаждения, рассматриваются возможности осуществления с помощью лазеров управляемого термоядерного синтеза. Лазеры используются и в военных целях, например, в качестве средств наведения и прицеливания. Рассматриваются варианты создания на основе мощных лазеров боевых систем защиты воздушного, морского и наземного базирования. question: Какое излучение позволяет обеспечить точную локализацию, дозированность, абсолютную стерильность и высокую скорость ввода энергии в систему?
лазерное
Лазерное


## 4. Файнтюн адаптера

В уроках про LLM вы изучили адаптеры, которые помогают уменьшить потребляемые во время обучения ресурсы. Поскольку LoRA встраивается в линейные слои, то такой подход применим и к encoder-decoder моделям. Дообучим под задачу теперь LoRA-адаптер для ruT5-base и сравним качество с предыдущими результатами. 


### Задание 4
Помимо кода из предыдущего задания, реализуйте добавление LoRA-адаптера так, чтобы процент обучаемых параметров не превышал 1.


In [46]:
import torch
from transformers import AutoModelForSeq2SeqLM, \
                        DataCollatorForSeq2Seq, \
                        Seq2SeqTrainingArguments, \
                        Seq2SeqTrainer
from peft import LoraConfig, get_peft_model

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
peft_config = LoraConfig(
    r=8, # ранг
    lora_alpha=16, # вес добавления адаптера
    lora_dropout=0.05,
    bias="none",
    target_modules=["q", "k", "v",],  # слои, к которым применяем 
    task_type="SEQ_2_SEQ_LM",
)
model = get_peft_model(model, peft_config)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="rut5_base_lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=1,
    eval_strategy="epoch",
    predict_with_generate=True,
    gradient_accumulation_steps=1,
    optim='adafactor', # специально придуманный оптимизатор для T5
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

model.print_trainable_parameters()

trainable params: 1,327,104 || all params: 224,230,656 || trainable%: 0.5918


In [47]:
trainer.train()

res = trainer.predict(tokenized_val)

preds = res.predictions
preds[preds < 0] = 0
pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
labels = np.array(tokenized_val['labels'])
labels[labels < 0] = 0
label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

print(compute_metrics_with_evaluate(pred_texts, label_texts))

TypeError: T5ForConditionalGeneration.forward() got an unexpected keyword argument 'num_items_in_batch'

In [48]:
class FixedSeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs.loss

        if return_outputs:
            return loss, outputs
        return loss

trainer = FixedSeq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

trainer.train()

res = trainer.predict(tokenized_val)

preds = res.predictions
preds[preds < 0] = 0
pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
labels = np.array(tokenized_val['labels'])
labels[labels < 0] = 0
label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

print(compute_metrics_with_evaluate(pred_texts, label_texts))

Epoch,Training Loss,Validation Loss


RuntimeError: 
            Some tensors share memory, this will lead to duplicate memory on disk and potential differences when loading them again: [{'base_model.model.encoder.embed_tokens.weight', 'base_model.model.decoder.embed_tokens.weight', 'base_model.model.shared.weight', 'base_model.model.lm_head.weight'}].
            A potential way to correctly save your model is to use `save_model`.
            More information at https://huggingface.co/docs/safetensors/torch_shared_tensors
            

In [ ]:
idx = 0
print(tokenizer.decode(tokenized_val[idx]["input_ids"], skip_special_tokens=True))
print(pred_texts[idx])
print(label_texts[idx])

context: Первые упоминания о строении человеческого тела встречаются в Древнем Египте. В XXVII веке до н. э. египетский врач Имхотеп описал некоторые органы и их функции, в частности головной мозг, деятельность сердца, распространение крови по сосудам. В древнекитайской книге Нейцзин (XI—VII вв. до н. э.) упоминаются сердце, печень, лёгкие и другие органы тела человека. В индийской книге Аюрведа ( Знание жизни , IX-III вв. до н. э.) содержится большой объём анатомических данных о мышцах, нервах, типах телосложения и темперамента, головном и спинном мозге. question: Где встречаются первые упоминания о строении человеческого тела?
в Древнем Египте
в Древнем Египте
